In [1]:
import os
os.chdir('../')
%pwd

'/home/minh_khai/salinity/salinity-ml'

In [2]:
from dataclasses import dataclass
from pathlib import Path

from core.models.configs import (
    ModelConfig,
    XGBoostConfig, LSTMConfig, iTransformerConfig
)

@dataclass(frozen=True)
class DataConfig:
    train_data_dir: Path
    valid_data_dir: Path

ModelSpecificConfig = XGBoostConfig | LSTMConfig | iTransformerConfig

@dataclass(frozen=True)
class TrainingConfig:
    root_dir:       Path
    model_dir:      Path
    encoders_dir:   Path
    
    model:  ModelConfig
    data:   DataConfig
    
    model_config:   ModelSpecificConfig

In [3]:
import os
from core.constants import *
from core import read_yaml, create_directories

class ConfigurationManager:
    
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    _CONFIG_BUILDERS = {
        'xgboost': '_get_xgboost_config',
        'lstm':    '_get_lstm_config',
        'iTransformer': '_get_itransformer_config',
    }
    
    def _get_model_config(self, model_name: str) -> ModelSpecificConfig:
        builder_name = self._CONFIG_BUILDERS.get(model_name)
        if not builder_name:
            raise ValueError(
                f"Unsupported model: '{model_name}'. "
                f"Choose from: {list(self._CONFIG_BUILDERS)}"
            )
        return getattr(self, builder_name)()
    
    def _get_xgboost_config(self) -> XGBoostConfig:
        p = self.params.xgboost
        return XGBoostConfig(
            n_estimators   = p.n_estimators,
            max_depth      = p.max_depth,
            learning_rate  = p.learning_rate,
            subsample      = p.subsample
        )
    
    def _get_lstm_config(self) -> LSTMConfig:
        p = self.params.lstm
        return LSTMConfig(
            hidden_size    = p.hidden_size,
            num_layers     = p.num_layers,
            dropout        = p.dropout,
            
            epochs         = p.epochs,
            batch_size     = p.batch_size,
            learning_rate  = p.learning_rate,
            patience       = p.patience,
        )
    
    def _get_itransformer_config(self) -> iTransformerConfig:
        p = self.params.iTransformer
        return iTransformerConfig(
            hidden_size    = p.hidden_size,
            epochs         = p.epochs,
            batch_size     = p.batch_size,
            learning_rate  = p.learning_rate,
            patience       = p.patience,
        )

    def get_training_config(self) -> TrainingConfig:
        config = self.config.training_config
        params = self.params.training_params
        
        model_dir = Path(config.root_dir) / params.model_name

        create_directories([
            config.root_dir,
            config.encoders_dir,
            model_dir,
        ])
        
        return TrainingConfig(
            root_dir        = Path(config.root_dir),
            model_dir       = model_dir, 
            encoders_dir    = Path(config.encoders_dir),
            
            model = ModelConfig(
                model_name      = params.model_name,
                window_size     = params.sliding_window_size,
                horizon         = params.sliding_horizon
            ),
            data = DataConfig(
                train_data_dir  = Path(self.config.transformation_config.root_dir) / "train",
                valid_data_dir  = Path(self.config.transformation_config.root_dir) / "valid",  
            ),
            
            model_config = self._get_model_config(model_name=params.model_name),
        )

In [4]:
from core.models import BaseModelStrategy
from core.models.xgboost_strategy import XGBoostStrategy
from core.models.lstm_strategy import LSTMStrategy
from core.models.iTransformer_strategy import iTransformerStrategy

STRATEGY_REGISTRY = {
    "xgboost": XGBoostStrategy,
    "lstm": LSTMStrategy,
    "iTransformer": iTransformerStrategy,
}

def get_strategy(config: TrainingConfig, **kwargs) -> BaseModelStrategy:
    name = config.model.model_name

    if name not in STRATEGY_REGISTRY:
        raise ValueError(f"Unsupported model: '{name}'. Choose from: {list(STRATEGY_REGISTRY)}")
    
    strategy_cls = STRATEGY_REGISTRY[name]
    
    if name == "xgboost":
        return strategy_cls(config.model_config)
    elif name == "lstm":
        return strategy_cls(config.model_config, config.model, **kwargs)
    elif name == "iTransformer":
        return strategy_cls(config.model_config, config.model)

In [5]:
import os, glob, json, time, joblib, mlflow
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import asdict

from core.logging import logger
from core.data.process import preprocess
from core.data.sliding_window import sliding

class Training:
    def __init__(self, config: TrainingConfig):
        self.config       = config
        self.model_config = config.model_config

        self.clean_df, self.train_cutoff = self._get_clean_df()
        self.X_train, self.y_train, self.X_valid, self.y_valid = self._get_windows()
        self.model = self._get_model()
        
        
    # ─────────────────────────────────────────────
    # Data
    # ─────────────────────────────────────────────
    def _get_clean_df(self):
        train_files = glob.glob(os.path.join(self.config.data.train_data_dir, "*.csv"))
        valid_files = glob.glob(os.path.join(self.config.data.valid_data_dir, "*.csv"))
        
        train_df = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
        valid_df = pd.concat([pd.read_csv(f) for f in valid_files], ignore_index=True)
        
        full_df = pd.concat([train_df, valid_df], ignore_index=True)
        full_df = full_df.sort_values("timestamp").reset_index(drop=True)
        
        train_cutoff = train_df["timestamp"].max()
        logger.info(f"Train cutoff: {train_cutoff}")
        
        clean_df = preprocess(full_df, out_dir=str(self.config.encoders_dir), fit_encoders=True)
        return clean_df, train_cutoff
    
    def _get_windows(self):
        logger.info("Preparing sliding windows...")
        return sliding(
            self.clean_df, self.train_cutoff,
            model_type  = self.config.model.model_name,
            window_size = self.config.model.window_size,
            horizon     = self.config.model.horizon,
        )
        
    
    # ─────────────────────────────────────────────
    # Model
    # ─────────────────────────────────────────────
    def _get_model(self) -> BaseModelStrategy:
        name = self.config.model.model_name
        if name == "lstm":
            return get_strategy(self.config, input_size=self.X_train.shape[-1])
        return get_strategy(self.config)
    
      
    # ─────────────────────────────────────────────
    # Train
    # ─────────────────────────────────────────────
    def train(self):
        mlflow.log_params(self._get_log_params())
        
        # ── training ─────────────────────────────────────────
        logger.info("Training model...")

        start   = time.time()
        self.model.train(self.X_train, self.y_train)
        elapsed = time.time() - start
        
        logger.info(f"Training done in {elapsed:.2f}s")
        
        # ── Validation ─────────────────────────────────
        metrics = self.model.validate(self.X_valid, self.y_valid)
        logger.info(f"Validation metrics: {json.dumps(metrics, indent=2)}")
        mlflow.log_metrics(metrics)
        
        # ── Saving weights ─────────────────────────────────
        self.model.save(self.config.model_dir)

        # ── Optim - ONNX ─────────────────────────────────
        self.model.save_onnx(self.config.model_dir)

        mlflow.log_artifacts(str(self.config.model_dir))
        return metrics
    
    
    # ─────────────────────────────────────────────
    # Helpers
    # ─────────────────────────────────────────────
    def _get_log_params(self) -> dict:
        return {
            "model_name":  self.config.model.model_name,
            "window_size": self.config.model.window_size,
            "horizon":     self.config.model.horizon,
            **{k: v for k, v in asdict(self.model_config).items()},
        }

/home/minh_khai/salinity/salinity-ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import sys
import mlflow
from core.exception import CustomException

try:
    cfg_manager = ConfigurationManager()
    config      = cfg_manager.get_training_config()
    training    = Training(config=config)
    
    with mlflow.start_run(run_name=config.model.model_name):
        metrics = training.train()
        print(metrics)

except Exception as e:
    raise CustomException(e, sys)

[2026-05-24 11:09:32,885: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-05-24 11:09:32,889: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-05-24 11:09:32,891: INFO: __init__: created directory at: artifacts]
[2026-05-24 11:09:32,893: INFO: __init__: created directory at: artifacts/training]
[2026-05-24 11:09:32,894: INFO: __init__: created directory at: artifacts/encoders]
[2026-05-24 11:09:32,896: INFO: __init__: created directory at: artifacts/training/lstm]
[2026-05-24 11:09:32,907: INFO: 757285498: Train cutoff: 2022-01-26]
[2026-05-24 11:09:32,928: INFO: 757285498: Preparing sliding windows...]
[2026-05-24 11:09:39,983: INFO: 757285498: Training model...]
Epoch 1 | Loss: 209.6259
Epoch 2 | Loss: 174.3891
Epoch 3 | Loss: 133.2234
Epoch 4 | Loss: 114.0577
Epoch 5 | Loss: 104.6736
Epoch 6 | Loss: 98.4730
Epoch 7 | Loss: 93.0757
Epoch 8 | Loss: 83.4044
Epoch 9 | Loss: 75.9694
Epoch 10 | Loss: 67.6816
Epoch 11 | Loss: 62.1985
Epoch 12 | Los

/home/minh_khai/salinity/salinity-ml/.venv/lib/python3.12/site-packages/torch/onnx/symbolic_opset9.py:4244: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(
